In [1]:
import pandas as pd
import os


In [2]:
NOTEBOOK_DIR= os.getcwd()
BASE_DIR= os.path.join(NOTEBOOK_DIR, "..", "..")
data= os.path.join(BASE_DIR, "data", "external", "pubmedqa", "pubmedqa.csv")
PROCESSED_DATA_DIR= os.path.join(BASE_DIR, "data","processed","pubmedqa")

In [3]:
df= pd.read_csv(data)
print(df.shape)
print(df.columns)


(273518, 5)
Index(['pubid', 'question', 'context', 'long_answer', 'final_decision'], dtype='object')


In [4]:
print(df.iloc[0]["pubid"])


25429730


In [5]:
print(df.iloc[0]["question"])


Are group 2 innate lymphoid cells ( ILC2s ) increased in chronic rhinosinusitis with nasal polyps or eosinophilia?


In [6]:
print(df.iloc[0]["context"])

{'contexts': ['Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.', 'The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease.', 'A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing pituitary tumour resection through transphenoidal approach. ILC2s were identified as CD45(+) Lin(-) CD127(+) CD4(-) CD8(-) CRTH2(CD294)(+) CD161(+) cells in single cell suspensions through flow cytometry. ILC2 frequencies, measured as a percentage of CD45(+) cells, were compared across CRS phenotype, endotype

In [7]:
print(df.iloc[0]["long_answer"])

As ILC2s are elevated in patients with CRSwNP, they may drive nasal polyp formation in CRS. ILC2s are also linked with high tissue and blood eosinophilia and have a potential role in the activation and survival of eosinophils during the Th2 immune response. The association of innate lymphoid cells in CRS provides insights into its pathogenesis.


In [8]:
print(df.iloc[0]["final_decision"])

yes


In [9]:
# parse string thành Python object
import ast
df["context"] = df["context"].apply(ast.literal_eval)

In [10]:
r= df.iloc[0]["context"].keys()
print(r)

dict_keys(['contexts', 'labels', 'meshes'])


In [11]:
print(df.iloc[0]["context"]["labels"])

['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS']


In [12]:
print(df.iloc[0]["context"]["meshes"])

['Adult', 'Aged', 'Antigens, Surface', 'Case-Control Studies', 'Chronic Disease', 'Eosinophilia', 'Female', 'Humans', 'Hypersensitivity', 'Immunity, Innate', 'Immunoglobulin E', 'Immunophenotyping', 'Leukocyte Count', 'Lymphocyte Subsets', 'Male', 'Middle Aged', 'Nasal Mucosa', 'Nasal Polyps', 'Neutrophil Infiltration', 'Patient Outcome Assessment', 'Rhinitis', 'Sinusitis', 'Young Adult']


In [13]:
result = " ".join(df.iloc[0]["context"]["contexts"])
print(result)

Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated. The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease. A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing pituitary tumour resection through transphenoidal approach. ILC2s were identified as CD45(+) Lin(-) CD127(+) CD4(-) CD8(-) CRTH2(CD294)(+) CD161(+) cells in single cell suspensions through flow cytometry. ILC2 frequencies, measured as a percentage of CD45(+) cells, were compared across CRS phenotype, endotype, inflammatory CRS su

In [14]:
print(df.iloc[0]["context"]["contexts"])


['Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.', 'The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease.', 'A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing pituitary tumour resection through transphenoidal approach. ILC2s were identified as CD45(+) Lin(-) CD127(+) CD4(-) CD8(-) CRTH2(CD294)(+) CD161(+) cells in single cell suspensions through flow cytometry. ILC2 frequencies, measured as a percentage of CD45(+) cells, were compared across CRS phenotype, endotype, inflammator

### Text Representation

Try to build document in 3 ways: contexts, contexts + labels, contexts + labels + meshes

In [24]:
def extract_context(context):
    return " ".join(context)

In [25]:
def split_text(text, chunk_size=250, overlap=50):
    words = text.split()
    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

#### Context only

In [26]:
import json

docs_context= []

for _, r in df.iterrows():

    doc_id = str(r["pubid"])
    text = extract_context(r["context"]["contexts"])

    chunks = split_text(text)

    for i, chunk in enumerate(chunks):
        docs_context.append({
            "doc_id": doc_id,
            "chunk_id": f"{doc_id}_{i}",
            "text": chunk,
            "source": "pubmed"
        })

OUTPUT_FILE= os.path.join(PROCESSED_DATA_DIR,"docs_context.json")

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(docs_context, f, indent=2)

#### Contexts + Labels

In [27]:
def join_context_label(contexts, labels):
    return "\n".join(
        f"{label}: {ctx}"
        for label, ctx in zip(labels, contexts)
    )

In [19]:
res= join_context_label(df.iloc[0]["context"]["contexts"], df.iloc[0]["context"]["labels"])
print(res)

BACKGROUND: Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.
OBJECTIVE: The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease.
METHODS: A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing pituitary tumour resection through transphenoidal approach. ILC2s were identified as CD45(+) Lin(-) CD127(+) CD4(-) CD8(-) CRTH2(CD294)(+) CD161(+) cells in single cell suspensions through flow cytometry. ILC2 frequencies, measured as a percentage of CD45(+) cells, were compared across CRS phenotyp

In [28]:
import json

docs_context_label = []

for _, r in df.iterrows():

    ctx = r["context"] 
    
    contexts = ctx["contexts"]
    labels = ctx["labels"]
    
    doc_id = str(r["pubid"])

    text = join_context_label(contexts, labels)

    chunks = split_text(text)

    for i, chunk in enumerate(chunks):
        docs_context_label.append({
            "doc_id": doc_id,
            "chunk_id": f"{doc_id}_{i}",
            "text": chunk,
            "source": "pubmed"

        })

OUTPUT_FILE_2= os.path.join(PROCESSED_DATA_DIR,"docs_context_label.json")

with open(OUTPUT_FILE_2, "w") as f:
    json.dump(docs_context_label, f, indent=2)

#### Context + Labels + Meshes

In [29]:
docs_context_label_mesh = []

for _, r in df.iterrows():

    ctx = r["context"]

    contexts = ctx["contexts"]
    labels = ctx["labels"]
    meshes = ctx["meshes"]

    doc_id = str(r["pubid"])

    # text từ context + label
    text = join_context_label(contexts, labels)

    # thêm MeSH vào text để retrieval hiểu semantic
    mesh_text = " ".join(meshes)
    full_text = text + " MeSH terms: " + mesh_text

    chunks = split_text(full_text)

    for i, chunk in enumerate(chunks):

        docs_context_label_mesh.append({
            "doc_id": doc_id,
            "chunk_id": f"{doc_id}_{i}",
            "text": chunk,
            "labels": labels,
            "meshes": meshes,
            "source": "pubmed"

        })



OUTPUT_FILE_3 = os.path.join(PROCESSED_DATA_DIR, "docs_context_label_mesh.json")

with open(OUTPUT_FILE_3, "w", encoding="utf-8") as f:
    json.dump(docs_context_label_mesh, f, indent=2, ensure_ascii=False)